# 🎙️ Golden Transcription Pipeline

## How this works (READ FIRST)

| Phase | Dataset | Purpose |
|-------|---------|-------- |
| **A — Training** | `validation_dataset.csv` (FLEURS, 692 rows, multilingual) | Train XGBoost to learn how to fuse scores |
| **B — Prediction** | `transcription_assessment.csv` (Arabic test set) | Use trained model to predict golden transcription |

⚠️ **The Arabic 100-row dataset is the TEST set — it is never used for training.**

## Before running:
1. **GPU**: Settings → Accelerator → **GPU T4 x2** ✅
2. **Internet**: Settings → Internet → **ON** ✅
3. **Add Data** (right panel) → add your Kaggle dataset containing:
   - `validation_dataset.csv`
   - `fleurs_audio/` folder
   - `transcription_assessment.csv` (or your Arabic test CSV)
   - `temp_audio/` folder (Arabic test audio files)
4. Edit **Cell 3** with the correct paths
5. Click **Run All**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
import subprocess, sys
pkgs = ['openai-whisper','jiwer','xgboost','python-Levenshtein',
        'transformers','sentencepiece','accelerate','protobuf']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('✅ Packages installed')

In [ ]:
# ── Cell 2: Clone repo (gets kaggle_runner.py) ─────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/Vinar01/Team-team'
DEST = '/kaggle/working/repo'

if not os.path.exists(DEST):
    subprocess.check_call(['git','clone','--depth','1', REPO, DEST])
else:
    subprocess.check_call(['git','-C', DEST,'pull'])

print('✅ Repo ready')

# Show dataset files so you can copy-paste paths into Cell 3
print('\n── Your Kaggle input files ──')
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for f in files:
        if f.endswith(('.csv','.xlsx')) or 'audio' in root.lower():
            print(' ', os.path.join(root, f))

In [ ]:
# ── Cell 3: ⚙️  EDIT THESE PATHS ──────────────────────────────────────────
# Copy paths printed by Cell 2 and paste here

# ── TRAINING DATA (FLEURS — has correct_option) ────────────────────────────
TRAIN_CSV   = '/kaggle/input/YOUR-DATASET/validation_dataset.csv'
TRAIN_AUDIO = '/kaggle/input/YOUR-DATASET/fleurs_audio'   # has en_us/, ar_eg/, ... subfolders

# ── TEST DATA (Arabic test set — predict on this) ──────────────────────────
# Use transcription_assessment.csv (the Arabic dataset)
# If your test CSV is somewhere else, update accordingly
TEST_CSV    = '/kaggle/input/YOUR-DATASET/transcription_assessment.csv'
TEST_AUDIO  = '/kaggle/input/YOUR-DATASET/temp_audio'     # flat folder: 1.wav, 2.wav ...

# ── MODEL & OUTPUT ─────────────────────────────────────────────────────────
MODEL_PATH  = '/kaggle/working/fusion_model.json'
OUTPUT_PATH = '/kaggle/working/submission.csv'

# ── SETTINGS ───────────────────────────────────────────────────────────────
WHISPER_MODEL = 'base'    # 'tiny' is fastest, 'small' is more accurate
FAST_MODE     = True      # True = Whisper+char only (~30 min)
                           # False = full pipeline with E5+mT5 (~2-3 hrs)
LIMIT = None              # Set e.g. 5 to test on 5 rows first

import torch
print(f'GPU: {"YES ✅" if torch.cuda.is_available() else "NO ❌ — go to Settings → Accelerator → GPU T4!"}')
print(f'Mode: {"FAST (Whisper+char)" if FAST_MODE else "FULL (Whisper+char+E5+mT5)"}')

In [ ]:
# ── Cell 4: Verify your paths and data before running ─────────────────────
import pandas as pd
from pathlib import Path

print('─── TRAINING DATA ───')
if Path(TRAIN_CSV).exists():
    df_train = pd.read_csv(TRAIN_CSV)
    print(f'  Rows: {len(df_train)}')
    print(f'  Columns: {list(df_train.columns)}')
    print(f'  Languages: {sorted(df_train["language"].unique()) if "language" in df_train.columns else "?"}')
    has_labels = any(c in df_train.columns for c in ('correct_option','true_golden_option_index'))
    print(f'  Labels: {"YES ✅" if has_labels else "NO ❌"}')
else:
    print(f'  ❌ File not found: {TRAIN_CSV}')

print(f'  Audio dir: {"EXISTS ✅" if Path(TRAIN_AUDIO).exists() else "NOT FOUND ❌"}')

print()
print('─── TEST DATA ───')
if Path(TEST_CSV).exists():
    df_test = pd.read_csv(TEST_CSV)
    print(f'  Rows: {len(df_test)}')
    print(f'  Columns: {list(df_test.columns)}')
else:
    print(f'  ❌ File not found: {TEST_CSV}')

print(f'  Audio dir: {"EXISTS ✅" if Path(TEST_AUDIO).exists() else "NOT FOUND ❌ (audio will be downloaded from URLs)"}')

In [ ]:
# ── Cell 5: 🚀 PHASE A — Train XGBoost on FLEURS data ─────────────────────
# Runs Whisper + char scoring on 692 FLEURS rows, trains XGBoost
# Takes ~25-35 minutes on GPU T4
import subprocess, sys

cmd_train = [
    sys.executable, '/kaggle/working/repo/kaggle_runner.py',
    '--mode',         'train',
    '--train-csv',    TRAIN_CSV,
    '--train-audio',  TRAIN_AUDIO,
    '--model-path',   MODEL_PATH,
    '--whisper-model', WHISPER_MODEL,
]
if not FAST_MODE:
    cmd_train.append('--full')
if LIMIT:
    cmd_train += ['--limit', str(LIMIT)]

print('Running Phase A (Training)...')
print(' '.join(cmd_train))
print('=' * 60)

proc = subprocess.Popen(cmd_train, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()

import os
if proc.returncode == 0 and os.path.exists(MODEL_PATH):
    size = os.path.getsize(MODEL_PATH) / 1024
    print(f'\n✅ Phase A done! XGBoost model saved ({size:.0f} KB): {MODEL_PATH}')
else:
    print(f'\n❌ Phase A exited with code {proc.returncode}')

In [ ]:
# ── Cell 6: 🚀 PHASE B — Predict on Arabic test set ────────────────────────
# Runs Whisper + char scoring on the test set, applies trained XGBoost
# Takes ~5-15 minutes on GPU T4 (depending on test set size)
import subprocess, sys

cmd_pred = [
    sys.executable, '/kaggle/working/repo/kaggle_runner.py',
    '--mode',         'predict',
    '--test-csv',     TEST_CSV,
    '--test-audio',   TEST_AUDIO,
    '--model-path',   MODEL_PATH,
    '--output',       OUTPUT_PATH,
    '--whisper-model', WHISPER_MODEL,
]
if not FAST_MODE:
    cmd_pred.append('--full')
if LIMIT:
    cmd_pred += ['--limit', str(LIMIT)]

print('Running Phase B (Prediction on test set)...')
print(' '.join(cmd_pred))
print('=' * 60)

proc = subprocess.Popen(cmd_pred, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()

if proc.returncode == 0:
    print('\n✅ Phase B done!')
else:
    print(f'\n❌ Phase B exited with code {proc.returncode}')

In [ ]:
# ── Cell 7: View results ────────────────────────────────────────────────────
import pandas as pd

results = pd.read_csv(OUTPUT_PATH)
print(f'Output: {len(results)} rows | {OUTPUT_PATH}')
print()

cols = ['audio_id','language','predicted_option_num','correct_option',
        'is_correct','score_1','score_2','score_3','score_4','score_5']
print(results[[c for c in cols if c in results.columns]].head(20).to_string(index=False))

if 'is_correct' in results.columns:
    ok  = int(results['is_correct'].sum())
    tot = int(results['is_correct'].notna().sum())
    print(f'\n{"="*50}')
    print(f'ACCURACY: {ok}/{tot}  ({100*ok/tot:.1f}%)')
    print(f'{"="*50}')
    print('\nPredicted option distribution:')
    print(results['predicted_option_num'].value_counts().sort_index())

print('\n✅ Download submission.csv from Output tab →')